# StSN Builder — structure-based (foldseek)

Builds **Structure Similarity Networks (StSN)** from a directory of PDB / mmCIF files using foldseek.  
For sequence-based networks (mmseqs / FASTA), see `build_ssn.ipynb`.

The `pident` column in foldseek output represents TM-score × 100, so thresholds work the same way as in SSNs — threshold 0.5 means TM-score ≥ 0.5.

Run cells top to bottom, skipping optional ones as needed.

## Setup

In [2]:
import sys, logging
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("stsn")

In [5]:
# Shared paths — edit these before running anything else
INPUT_PDB_DIR = Path("../data/structures/")    # directory of .pdb / .cif files
PREFIX        = "stsn_test"
OUTPUT_DIR    = Path("../results/stsn_run")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## Step 1 — Redundancy reduction *(optional)*

Run **one** of the two cells below.

### 1a · Cluster with foldseek

In [7]:
from foldseek import easy_cluster

result = easy_cluster(
    pdb_dir        = INPUT_PDB_DIR,
    output_dir     = OUTPUT_DIR / "cluster",
    prefix         = PREFIX,
    min_tmscore    = 0.5,    # TM-score threshold for cluster membership (0–1)
    coverage       = 0.8,
    cov_mode       = 0,      # 0 = bidirectional, 1 = query, 2 = target
    alignment_type = 2,      # 0 = 3Di, 1 = TM-align, 2 = 3Di+AA
    log            = log,
)
rep_pdb_dir = INPUT_PDB_DIR   # foldseek clusters by ID; use same dir with rep list
cluster_tsv = result["cluster_tsv"]
print(f"cluster_tsv: {cluster_tsv}")

13:37:37  INFO      CMD  foldseek easy-cluster ../data/structures ../results/stsn_run/cluster/stsn_test ../results/stsn_run/cluster/tmp --min-seq-id 0.5 --cov-mode 0 -c 0.8 --alignment-type 2
13:37:41  INFO           Create directory ../results/stsn_run/cluster/tmp
13:37:41  INFO           easy-cluster ../data/structures ../results/stsn_run/cluster/stsn_test ../results/stsn_run/cluster/tmp --min-seq-id 0.5 --cov-mode 0 -c 0.8 --alignment-type 2 
13:37:41  INFO           
13:37:41  INFO           MMseqs Version:                     	10.941cd33
13:37:41  INFO           Substitution matrix                 	aa:3di.out,nucl:3di.out
13:37:41  INFO           Seed substitution matrix            	aa:3di.out,nucl:3di.out
13:37:41  INFO           Sensitivity                         	4
13:37:41  INFO           k-mer length                        	0
13:37:41  INFO           Target search mode                  	0
13:37:41  INFO           k-score                             	seq:2147483647,prof:21474

cluster_tsv: ../results/stsn_run/cluster/stsn_test_cluster.tsv


### 1b · Skip clustering — use full PDB directory

In [ ]:
rep_pdb_dir = INPUT_PDB_DIR
cluster_tsv = None
print(f"Using full PDB directory: {rep_pdb_dir}")

---
## Step 2 — All-vs-all structure similarity search (foldseek)

Run **one** of the two cells below.

### 2a · Run foldseek easy-search

In [8]:
from foldseek import easy_search

result = easy_search(
    pdb_dir        = rep_pdb_dir,
    output_dir     = OUTPUT_DIR / "search",
    prefix         = PREFIX,
    alignment_type = 2,    # 0 = 3Di, 1 = TM-align, 2 = 3Di+AA
    log            = log,
)
m8_path = result["m8"]
print(f"m8: {m8_path}")

13:37:50  INFO      CMD  foldseek easy-search ../data/structures ../data/structures ../results/stsn_run/search/stsn_test.m8 ../results/stsn_run/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits --alignment-type 2
13:37:52  INFO           Create directory ../results/stsn_run/search/tmp
13:37:52  INFO           easy-search ../data/structures ../data/structures ../results/stsn_run/search/stsn_test.m8 ../results/stsn_run/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits --alignment-type 2 
13:37:52  INFO           
13:37:52  INFO           MMseqs Version:                    	10.941cd33
13:37:52  INFO           Seq. id. threshold                 	0
13:37:52  INFO           Coverage threshold                 	0
13:37:52  INFO           Coverage mode                      	0
13:37:52  INFO           Max reject                         	2147483647
13:37:52  INFO           Max accept 

m8: ../results/stsn_run/search/stsn_test.m8


### 2b · Use existing .m8 file

In [ ]:
m8_path = Path("../results/my_previous_run/search/my_run.m8")
print(f"Using existing m8: {m8_path}")

---
## Load similarity data

In [9]:
from plot_ssn import load_data

df = load_data(m8_path)
df.head()

13:38:45  INFO      NumExpr defaulting to 8 threads.
13:38:46  INFO      START load similarity table ../results/stsn_run/search/stsn_test.m8
13:38:46  INFO      END   load similarity table ../results/stsn_run/search/stsn_test.m8: 0.00s
13:38:46  INFO      Loaded 361 rows from ../results/stsn_run/search/stsn_test.m8
13:38:46  INFO      START remove self-hits
13:38:46  INFO      END   remove self-hits: 0.00s
13:38:46  INFO      Removed self-hits where qseqid == sseqid: 19 rows
13:38:46  INFO      After removing self-hits: 342 rows


Loaded data: 342 edges


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
1,A0A0H4VKH8,A0A0H5SFF4,59.2,324,131,0,55,376,4,327,9.249000e-51,2201
2,A0A0H4VKH8,A0A0H4VMQ7,31.0,314,197,0,59,372,24,309,1.653000e-28,1051
3,A0A0H4VKH8,A0A0I9RRI3,24.7,350,238,0,27,376,1,317,8.228000e-23,702
4,A0A0H4VKH8,A0A0H4VUR7,17.7,317,231,0,59,375,1,282,1.643000e-16,514
5,A0A0H4VKH8,A0A0H5SY45,16.6,318,252,0,57,374,5,307,1.160000e-15,438


---
## Load annotations *(optional)*

Skip this cell to produce uncolored networks.  
Structure IDs in the m8 file are the PDB filenames without extension (e.g. `1abc` for `1abc.pdb`).

In [10]:
import pandas as pd
from plot_ssn import build_color_lookup, _read_meta

META_FILE  = Path("../data/GH43_full.tsv")  # TSV with a column matching PDB IDs
ID_COL     = "pdb_id"       # column in META_FILE matching the structure IDs
COLOR_COLS = ["family"]     # one plot is produced per column

meta = _read_meta(META_FILE, ID_COL)
print(f"Metadata: {len(meta)} rows  |  columns: {list(meta.columns)}")

color_lookups = {}
for col in COLOR_COLS:
    color_lookups[col] = build_color_lookup(meta, ID_COL, col, clusters=None)
    print(f"  '{col}': {len(color_lookups[col])} annotated entries")

13:39:09  WARNING   Missing metadata or color column 'family' not found. All nodes will be 'Unknown'.


Metadata: 63591 rows  |  columns: ['Family', 'Domain', 'Species', 'GenBank', 'Source']
  'family': 0 annotated entries


---
## Step 3a — Structure Similarity Network (StSN)

Full graph — all edges above the TM-score threshold.

In [11]:
import json
from plot_ssn import create_graph, exclude_singleton_components, get_layout, plot_ssn, graph_stats

THRESHOLDS         = [0.5, 0.7]       # TM-score thresholds (0–1)
NODE_SIZE          = 5
LAYOUT             = "component_grid" # "component_grid" | "fr" | "lgl" | "kk"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
stsn_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = create_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = exclude_singleton_components(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} edges")
    layout_coords = get_layout(g, LAYOUT)
    stsn_stats.append(graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_stsn_t{thresh}_{col_lbl}.html"
        plot_ssn(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_stsn_stats.json"
stats_path.write_text(json.dumps(stsn_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(stsn_stats)

13:39:16  INFO      START deduplicate undirected edges
13:39:16  INFO      Deduplicated reciprocal/parallel edges: 4 -> 2
13:39:16  INFO      END   deduplicate undirected edges: 0.01s
13:39:16  INFO      START create edge list
13:39:16  INFO      END   create edge list: 0.00s
13:39:16  INFO      START build igraph
13:39:16  INFO      END   build igraph: 0.00s
13:39:16  INFO      START attach edge pident weights
13:39:16  INFO      END   attach edge pident weights: 0.00s
13:39:16  INFO      START compute component_grid layout (4 nodes, 2 edges)
13:39:16  INFO      END   compute component_grid layout (4 nodes, 2 edges): 0.00s


t=0.5: 4 nodes, 2 edges


KeyError: 'pdb_id'

---
## Step 3b — Minimum Spanning Tree (StSN-MST) *(optional)*

Reduces each connected component to its spanning tree — keeps only the highest-TM-score edge per pair.  
Often the preferred view for structure networks as it avoids edge clutter.

In [ ]:
from plot_mst import (
    build_mst_graph,
    exclude_singleton_components as mst_exclude_singletons,
    get_layout as mst_get_layout,
    plot_mst,
    graph_stats as mst_graph_stats,
)

THRESHOLDS         = [0.5, 0.7]       # TM-score thresholds (0–1)
NODE_SIZE          = 5
LAYOUT             = "component_grid"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
mst_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = build_mst_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = mst_exclude_singletons(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} MST edges")
    layout_coords = mst_get_layout(g, LAYOUT)
    mst_stats.append(mst_graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_t{thresh}_{col_lbl}.html"
        plot_mst(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_stats.json"
stats_path.write_text(json.dumps(mst_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(mst_stats)

---
## Results

In [ ]:
from IPython.display import display, HTML

html_files = sorted((OUTPUT_DIR / "plots").glob("*.html"))
items = "".join(f'<li><a href="{f.resolve()}" target="_blank">{f.name}</a></li>' for f in html_files)
display(HTML(f"<ul>{items}</ul>") if items else HTML("<i>No plots found — run cells above first.</i>"))